# Create an Amazon Bedrock Knowledge Base

This notebook demonstrates how to create and configure an Amazon Bedrock Knowledge Base. 

The Knowledge Base integrates Amazon S3 as the data source for onboarding documents and uses Amazon OpenSearch Serverless as the vector store. It enables retrieval-augmented generation (RAG) by powering intelligent queries over structured and unstructured HR content like onboarding guides, policies, and FAQs.

If you want to create the Knowledge Base with your own documents, change the documents uploaded in the **Create the Amazon S3 bucket and upload the sample documents** section. This component will be used in the next notebook of this folder.

## Setup and prerequisites

### Prerequisites
* Complete the prerequisites notebook located in this folder
* Python 3.10+
* AWS account
* Amazon Nova Micro enabled on Amazon Bedrock
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket, Amazon OpenSearch Serverless

Let's now install the requirement packages and define the needed clients to create our Amazon Bedrock Knowledge Base:

In [1]:
%pip install -r requirements.txt

INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-text-splitters to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-aws to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-aws to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctr

In [1]:
import os
import json
import time
import uuid
import boto3
import logging
import requests
from datetime import datetime

In [2]:
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
bedrock_agent_runtime_client = boto3.client("bedrock-agent-runtime",region_name=region)

In [3]:
os.environ['AWS_DEFAULT_REGION'] = region

In [4]:
# Get the current timestamp
current_time = time.time()
# Format the timestamp as a string
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))[-7:]
# Create the suffix using the timestamp
suffix = f"{timestamp_str}"

## Download Amazon Bedrock Knowledge Bases helper
To expedite the knowledge base configuration and creation we will be downloading the knowledge base utility file. This contains a helper to generate knowledge bases abstracting the multiple API calls that need to be used.

In [6]:
url = "https://raw.githubusercontent.com/aws-samples/amazon-bedrock-samples/main/rag/knowledge-bases/features-examples/utils/knowledge_base.py"
target_path = "utils/knowledge_base.py"
os.makedirs(os.path.dirname(target_path), exist_ok=True)
response = requests.get(url)
with open(target_path, "w") as f:
    f.write(response.text)
print(f"Downloaded Knowledge Bases utils to {target_path}")

Downloaded Knowledge Bases utils to utils/knowledge_base.py


In [5]:
from utils.knowledge_base import BedrockKnowledgeBase

## Create Amazon Bedrock Knowledge Base
In this section we will configure the Amazon Bedrock Knowledge Base containing the policy documents andn FAQs for employee onboarding. We will be using Amazon Opensearch Serverless Service as the underlying vector store and Amazon S3 as the data source containing the files.

In [6]:
knowledge_base_name = f"aws-support-qa-knowledge-base0318"
knowledge_base_description = "AWS Support Agent Knowledge Base containing common QA document."
foundation_model = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

In [7]:
print(knowledge_base_name)

aws-support-qa-knowledge-base0318


In [8]:
param_kb_id = "/support/knowledge_base/kb_id"   # 建议用层级路径

For this notebook, we'll create a Knowledge Base with an Amazon S3 data source.

In [9]:
data_bucket_name = f'bedrock-aws-support-rag-bucket0318' # replace it with your first bucket name.
data_sources=[{"type": "S3", "bucket_name": data_bucket_name}]

### Create the Amazon S3 bucket and upload the sample documents
For this notebook, we'll create a Knowledge Base with an Amazon S3 data source.

In [12]:
import botocore
import os

def create_s3_bucket(bucket_name, region=None):
    s3 = boto3.client('s3', region_name=region)

    try:
        if region is None or region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
        print(f"✅ Bucket '{bucket_name}' created successfully.")
    except botocore.exceptions.ClientError as e:
        print(f"❌ Failed to create bucket: {e.response['Error']['Message']}")

create_s3_bucket(data_bucket_name, region)

✅ Bucket 'bedrock-aws-support-rag-bucket0318' created successfully.


In [10]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                file_to_upload = os.path.join(root,file)
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload,bucket_name,file)

In [11]:
upload_directory("./dataset", data_bucket_name)

uploading file ./dataset/FOOB申请流程.pdf to bedrock-aws-support-rag-bucket0318
uploading file ./dataset/Troubleshooting Tools.pdf to bedrock-aws-support-rag-bucket0318
uploading file ./dataset/ec2-network-pps-limit.csv to bedrock-aws-support-rag-bucket0318


### Create the Knowledge Base
We are now going to create the Knowledge Base using the abstraction located in the helper function we previously downloaded.

In [12]:
knowledge_base = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name}',
    kb_description=f'{knowledge_base_description}',
    data_sources=data_sources,
    chunking_strategy = "SEMANTIC", 
    suffix = f'{suffix}-f'
)

[2026-03-18 07:38:23,340] p1883 {credentials.py:1352} INFO - Found credentials in shared credentials file: ~/.aws/credentials
[2026-03-18 07:38:23,460] p1883 {credentials.py:1352} INFO - Found credentials in shared credentials file: ~/.aws/credentials


Step 1 - Creating or retrieving S3 bucket(s) for Knowledge Base documents
['bedrock-aws-support-rag-bucket0318']
buckets_to_check:  ['bedrock-aws-support-rag-bucket0318']
Bucket bedrock-aws-support-rag-bucket0318 already exists - retrieving it!
Step 2 - Creating Knowledge Base Execution Role (AmazonBedrockExecutionRoleForKnowledgeBase_8073753-f) and Policies
Step 3a - Creating OSS encryption, network and data access policies
Step 3b - Creating OSS Collection (this step takes a couple of minutes to complete)
{ 'ResponseMetadata': { 'HTTPHeaders': { 'connection': 'keep-alive',
                                         'content-length': '320',
                                         'content-type': 'application/x-amz-json-1.0',
                                         'date': 'Wed, 18 Mar 2026 07:38:24 '
                                                 'GMT',
                                         'x-amzn-requestid': 'e40fbb80-af76-4816-82ed-2a75d1dc516d'},
                        'HTTP

[2026-03-18 07:39:55,474] p1883 {base.py:258} INFO - PUT https://07go4uej5b631ibz63gl.us-east-1.aoss.amazonaws.com:443/bedrock-sample-rag-index-8073753-f [status:200 request:0.438s]



Creating index:
{ 'acknowledged': True,
  'index': 'bedrock-sample-rag-index-8073753-f',
  'shards_acknowledged': True}
Step 4 - Will create Lambda Function if chunking strategy selected as CUSTOM
Not creating lambda function as chunking strategy is SEMANTIC
Step 5 - Creating Knowledge Base
{ 'createdAt': datetime.datetime(2026, 3, 18, 7, 40, 55, 599166, tzinfo=tzlocal()),
  'description': 'AWS Support Agent Knowledge Base containing common QA '
                 'document.',
  'knowledgeBaseArn': 'arn:aws:bedrock:us-east-1:887221633712:knowledge-base/LJSA6IASTT',
  'knowledgeBaseConfiguration': { 'type': 'VECTOR',
                                  'vectorKnowledgeBaseConfiguration': { 'embeddingModelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v2:0'}},
  'knowledgeBaseId': 'LJSA6IASTT',
  'name': 'aws-support-qa-knowledge-base0318',
  'roleArn': 'arn:aws:iam::887221633712:role/AmazonBedrockExecutionRoleForKnowledgeBase_8073753-f',
  'status': 'CREATING',


### Start ingestion job
Once the KB and data source created, we can start the ingestion job for the data source. During the ingestion job, KB will fetch the documents in the data source, pre-process it to extract text, chunk it based on the chunking size provided, create embeddings of each chunk and then write it to the vector database, in this case OSS.

In [13]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base.start_ingestion_job()
# keep the kb_id for invocation later in the invoke request
kb_id = knowledge_base.get_knowledge_base_id()

job 1 started successfully

{ 'dataSourceId': 'ROF1BOWP3F',
  'ingestionJobId': 'LVWGN1ZEIV',
  'knowledgeBaseId': 'LJSA6IASTT',
  'startedAt': datetime.datetime(2026, 3, 18, 7, 41, 44, 439931, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 3,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 0,
                  'numberOfModifiedDocumentsIndexed': 0,
                  'numberOfNewDocumentsIndexed': 3},
  'status': 'COMPLETE',
  'updatedAt': datetime.datetime(2026, 3, 18, 7, 42, 3, 321679, tzinfo=tzlocal())}
'LJSA6IASTT'............................


### Test the Knowledge Base
We can now test the Knowledge Base to verify the documents have been ingested properly.

In [14]:
query = "what's the process of FOOB?"

In [15]:
#foundation_model = "amazon.nova-micro-v1:0"
foundation_model = "anthropic.claude-3-haiku-20240307-v1:0"

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id,
            "modelArn": "arn:aws:bedrock:{}::foundation-model/{}".format(region, foundation_model),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

The process for submitting a FOOB (External Capacity Request) is as follows:

1. Fill out the "Enter ticket details" section on the FOOB website, including the title, account ID, number of instances, and other details about the request. 2. Confirm the changes with the approver and update the CSV file in Maitre D if the changes are approved. Wait for the approver's approval before notifying the customer. 3. If the changes are not approved, confirm with the customer whether they need to submit a new request. 4. After the FOOB is approved, the customer will receive a notification in PHD to confirm. If the auto-accept feature is enabled, no manual confirmation is needed.



In [16]:
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

In [17]:
response_standard = response['citations'][0]['retrievedReferences']
print("# of citations or chunks used to generate the response: ", len(response_standard))
citations_rag_print(response_standard)

# of citations or chunks used to generate the response:  1
Chunk 1:  FOOB申请流程 ● 
 
 ● 
 
 ● 
 
 ○ 
 
 ○ 
 
 ○ 
 
 ■ 
 
  https://w.amazon.com/bin/view/EC2_Capacity_Planning_-_External_Capacity_Runbook/#HRunbook 
 
 FOOB是⽤来进⾏资源申请的流程，例如shein为了⿊五⼤促需要⼤量EC2资源，这就需要通过FOOB来申请 
 
 这⾥已SHEIN的⿊五FOOB申请为例，介绍⼀下对应流程 
 
 打开FOOB网站：  https://t.corp.amazon.com/D29794246/overview 
 
 只需要填写 “Enter ticket details”部分即可 
 
 Title部分⽰例 
 
 [r5b] [10/15]EC2 External FOOB Request - Customer Name: Zoetop(Shein)| Event: Black Friday 2021 | Region: PDX| Instance: c5/m5/r5/r5b| Need-By-Date: 2021-10-15 
 
 ■ 
 
 □ 
 
 □ 
 
 Description部分⽰例 
 
 account ID：填写所有需要申请资源的account id 
 
 number of instances: 按照每个account

Chunk 1 Location:  {'s3Location': {'uri': 's3://bedrock-aws-support-rag-bucket0318/FOOB申请流程.pdf'}, 'type': 'S3'}

Chunk 1 Metadata:  {'x-amz-bedrock-kb-source-uri': 's3://bedrock-aws-support-rag-bucket0318/FOOB申请流程.pdf', 'x-amz-bedrock-kb-source-file-modality': 'TEXT', 'x-amz-bedrock-kb-document-page-number':

### Store the Knowledge Base ID
Store the ID of the generated Knowledge Base in parametre store.

In [18]:
ssm = boto3.client("ssm",region_name=region)
response = ssm.put_parameter(
    Name=param_kb_id,
    Value=kb_id,
    Type="String",
    Overwrite=True  # True = 已存在就更新
)
print("Parameter stored:", response)

Parameter stored: {'Version': 2, 'Tier': 'Standard', 'ResponseMetadata': {'RequestId': '78a7f820-4f1f-410c-8a29-4d186afb7d18', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Wed, 18 Mar 2026 07:44:15 GMT', 'content-type': 'application/x-amz-json-1.1', 'content-length': '31', 'connection': 'keep-alive', 'x-amzn-requestid': '78a7f820-4f1f-410c-8a29-4d186afb7d18', 'cache-control': 'no-store'}, 'RetryAttempts': 0}}
